<a href="https://colab.research.google.com/github/daria-bazaliy/Kaggle-competitions/blob/main/rent_price_prediction_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
from datetime import datetime
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from xgboost import XGBRegressor


In [ ]:
def_train_path = "data/rent_train_preprocessed.xlsx"
def_test_path = "data/rent_test_preprocessed.xlsx"


def read_datasets() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_excel(def_train_path)
    test = pd.read_excel(def_test_path)

    return train, test


In [ ]:
def extract_city(s: str):
    if s.startswith('Свердловская область, Екатеринбург'):
        return 'Екатеринбург'
    if s.startswith('Санкт-Петербург'):
        return 'Санкт-Петербург'
    if s.startswith('Москва'):
        return 'Москва'
    raise Exception()


def tr(div):
    # return lambda x: (np.atan(x / div) / np.pi) * 2
    return lambda x: x


def do_map(ds: pd.DataFrame, l):
    res = []
    for _, row in ds.iterrows():
        res.append(l(row))
    return res


m = {
    'Екатеринбург': (56.8297381, 60.6042681),
    'Санкт-Петербург': (59.9332945, 30.3434109),
    'Москва': (55.7520251, 37.6184144),
}


def calc_dist(row):
    lat = row['lat']
    lon = row['lon']
    city = extract_city(row['Адрес'])
    if pd.isna(lat) or pd.isna(lon):
        return np.nan
    city_lat, city_lon = m[city]
    dist = math.sqrt(math.pow(city_lat - lat, 2) + math.pow(city_lon - lon, 2))
    return dist


In [ ]:
transformers = {
    'Комнаты': lambda ds, name: {
        name: ds[name].map(tr(4)),
    },
    'Площадь_общая': lambda ds, name: {
        name: ds[name].map(tr(100)),
    },
    'Этаж': lambda ds, name: {
        'Этаж_номер': ds[name].map(tr(20)),
        'Этажность': ds[name].map(tr(30)),
        'Этаж_первый': ds[name].map(lambda s: 1 if s == 1 else 0),
        'Этаж_последний': do_map(ds, lambda row: row['Этаж'] == row['Этажность']),
    },
    # 'Метро_станция': lambda ds, name: {
    #     'Метро_станция': ds[name].map(lambda s: 'undef' if pd.isna(s) else s),
    # },
    'Адрес': lambda ds, name: {
        'Город': ds[name].map(extract_city),
    },
    'Метро_минуты': lambda ds, name: {
        'Метро_минуты': ds[name].map(tr(20)),
    },
    'Метро_тип': lambda ds, name: {
        'Метро_тип': ds[name].map(lambda s: 'undef' if pd.isna(s) else s),
    },
    'lat': lambda ds, name: {
        'lat': ds[name],
        'lon': ds['lon'],
        'До_центра': do_map(ds, calc_dist),
    },
    'Дом': lambda ds, name: {
        'Дом_тип': ds[name].map(
            lambda s: 'undef' if (pd.isna(s) or ',' not in s) else s.split(', ')[1]
        ),
    },
    'Парковка': lambda ds, name: {
        name: ds[name].map(lambda s: 'undef' if pd.isna(s) else s),
    },
    'Ремонт': lambda ds, name: {
        name: ds[name].map(lambda s: 'undef' if pd.isna(s) else s),
    },
    'Балкон': lambda ds, name: {
        'Балконы': ds[name].map(
            lambda s: 0 if (pd.isna(s) or 'Балкон' not in s) else tr(5)(int(s.split('Балкон (')[1][:1]))
        ),
        'Лоджии': ds[name].map(
            lambda s: 0 if (pd.isna(s) or 'Лоджия' not in s) else tr(5)(int(s.split('Лоджия (')[1][:1]))
        ),
    },
    'Окна': lambda ds, name: {
        name: ds[name].map(lambda s: 'undef' if pd.isna(s) else s),
    },
    'Санузел': lambda ds, name: {
        'Санузел_совм': ds[name].map(
            lambda s: 0 if (pd.isna(s) or 'Совмещенный' not in s) else tr(5)(int(s.split('Совмещенный (')[1][:1]))
        ),
        'Санузел_разд': ds[name].map(
            lambda s: 0 if (pd.isna(s) or 'Раздельный' not in s) else tr(5)(int(s.split('Раздельный (')[1][:1]))
        ),
    },
    'Можно с детьми/животными': lambda ds, name: {
        'Дети': ds[name].map(
            lambda s: 0 if (pd.isna(s) or 'Можно с детьми' not in s) else 1
        ),
        'Животные': ds[name].map(
            lambda s: 0 if (pd.isna(s) or 'Можно с животными' not in s) else 1
        ),
    },
    'Высота потолков, м': lambda ds, name: {
        'Высота потолков': ds[name].map(lambda s: tr(3)(s if (pd.isna(s) or s < 10) else (s / 10 if s < 100 else s / 100))),
    },
    'Лифт': lambda ds, name: {
        'Лифт_пасс': ds[name].map(
            lambda s: 0 if (pd.isna(s) or 'Пасс' not in s) else tr(6)(int(s.split('Пасс (')[1][:1]))
        ),
        'Лифт_груз': ds[name].map(
            lambda s: 0 if (pd.isna(s) or 'Груз' not in s) else tr(4)(int(s.split('Груз (')[1][:1]))
        ),
    },
    'Мусоропровод': lambda ds, name: {
        name: ds[name].map(lambda s: 'undef' if pd.isna(s) else s),
    },
}


In [ ]:
all_column_names = []
included_column_names = []


def normalize_dataset(train: pd.DataFrame, test: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    def preprocess(ds: pd.DataFrame) -> pd.DataFrame:
        new_pds = pd.DataFrame()
        for name, transformer in transformers.items():
            for key, value in transformer(ds, name).items():
                new_pds[key] = value
        return new_pds

    train = preprocess(train)
    test = preprocess(test)

    encoders = {}
    for column, t in train.dtypes.items():
        if t == 'object':
            encoders[column] = LabelBinarizer()

    for column, col_encoder in encoders.items():
        col_encoder.fit(train[column])

    def normalize(ds: pd.DataFrame) -> pd.DataFrame:
        for name, encoder in encoders.items():
            new = encoder.transform(ds[name])
            new_df = pd.DataFrame(new, columns=[f'{name}_{cls}' for cls in encoder.classes_])
            ds = pd.concat([ds.drop(name, axis=1), new_df], axis=1)

        global all_column_names
        if len(all_column_names) == 0:
            all_column_names = list(ds.columns)
        for column_name in ds.columns:
            if column_name not in included_column_names:
                ds = ds.drop(column_name, axis=1)
        return ds

    return normalize(train), normalize(test)


In [ ]:
def prepare_datasets(
        train: pd.DataFrame,
        submission: pd.DataFrame,
        show: bool = False,
        draw: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    y_train = train['Price']
    # y_train = y_train.map(lambda p: min(p, 1e6))
    id_submission = submission['ID  объявления']

    x_train, x_submission = normalize_dataset(
        train,
        submission,
    )

    if show:
        print(x_train.dtypes)
        print()
        with pd.option_context('display.max_columns', 40):
            print(x_train.describe(include='all'))
            print()
        with pd.option_context('display.max_columns', 40):
            print(y_train.describe(include='all', percentiles=[0.1, 0.2, 0.5, 0.7, 0.9, 0.95, 0.99]))
            print()
    if draw:
        for column in x_train.columns:
            sns.displot(x_train[column], label=column)
            plt.show()
        sns.displot(y_train)
        plt.show()

    return x_train, y_train, x_submission, id_submission


In [ ]:
def test_models(train_test_sets) -> Tuple[str, float]:
    results = []
    # print(f'iterations: {len(models)}')
    # for index in tqdm(range(len(models))):

    for name, st in train_test_sets.items():
        model, x_train, x_test, y_train, y_test, _, _ = st
        # print(f'model: {model.name()}')
        model.fit(x_train, y_train)
        predictions = model.predict(x_test)

        error = mean_absolute_error(y_test, predictions)
        results.append([name, model.name(), error])

        print(f'{name}: error = {error:.6f}')
    print()

    results.sort(key=lambda r: (r[1], r[2]))
    for name, _, error in results:
        print(f'{name}: error = {error:.6f}')

    results.sort(key=lambda r: r[2])
    name, _, error = results[0]
    print()

    print(f'best: {name}, error = {error:.4f}')
    return name, error


In [ ]:
def write_best(model, error, x_submission, id_submission):
    predictions = model.predict(x_submission)
    submission = pd.DataFrame(
        data=[[int(id_submission[i]), predictions[i]] for i in range(len(id_submission))],
        columns=['ID', 'Price'],
    )
    now = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")
    submission.to_csv(
        f'sub_{error:.0f}_{now}_{model.name()}.csv',
        float_format='%.2f',
        index=False,
    )
    return


In [ ]:
class RandomForestRegressorModel:
    def __init__(self, n_estimators, max_depth, min_samples_leaf):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.model = RandomForestRegressor(
            random_state=127,
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
        )

    def name(self):
        return f'random_forest_{self.n_estimators}_{self.max_depth}_{self.min_samples_leaf}'

    def fit(self, x_train, y_train):
        self.model.fit(x_train, y_train)

    def predict(self, x_test):
        return self.model.predict(x_test)


In [ ]:
class DecisionTreeRegressorModel:
    def __init__(self, criterion, max_depth, min_samples_leaf):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.model = DecisionTreeRegressor(
            random_state=127,
            criterion=self.criterion,
            max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf,
        )

    def name(self):
        return f'decision_tree_{self.criterion}_{self.max_depth}_{self.min_samples_leaf}'

    def fit(self, x_train, y_train):
        self.model.fit(x_train, y_train)

    def predict(self, x_test):
        return self.model.predict(x_test)


In [ ]:
class ExtraTreeRegressorModel:
    def __init__(self, criterion, max_depth, min_samples_leaf):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.model = ExtraTreeRegressor(
            random_state=127,
            criterion=self.criterion,
            max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf,
        )

    def name(self):
        return f'extra_tree_{self.criterion}_{self.max_depth}_{self.min_samples_leaf}'

    def fit(self, x_train, y_train):
        self.model.fit(x_train, y_train)

    def predict(self, x_test):
        return self.model.predict(x_test)


In [ ]:
class GradientBoostingRegressorModel:
    def __init__(self, criterion, n_estimators, learning_rate, max_depth, min_samples_leaf):
        self.criterion = criterion
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.model = GradientBoostingRegressor(
            random_state=127,
            criterion=criterion,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
        )

    def name(self):
        return f'gradient_boosting_{self.criterion}_{self.n_estimators}_{self.learning_rate}_{self.max_depth}_{self.min_samples_leaf}'

    def fit(self, x_train, y_train):
        self.model.fit(x_train, y_train)

    def predict(self, x_test):
        return self.model.predict(x_test)


In [ ]:
class XGBoostingRegressorModel:
    def __init__(self, booster, n_estimators, learning_rate, max_depth):
        self.booster = booster
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.model = XGBRegressor(
            random_state=127,
            booster=booster,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
        )

    def name(self):
        return f'xgboost_{self.booster}_{self.n_estimators}_{self.learning_rate}_{self.max_depth}'

    def fit(self, x_train, y_train):
        self.model.fit(x_train, y_train)

    def predict(self, x_test):
        return self.model.predict(x_test)


In [ ]:
included_column_names = [
    'Площадь_общая',
    'lat',
    'lon',
    # 'Лоджии',
]

train, submission = read_datasets()
raw_ds_xs, _, _, _ = prepare_datasets(train, submission, True, False)

add_candidates = set()
# add_candidates = set(['nothing' if name in included_column_names else name for name in all_column_names])
add_candidates.add('nothing')

def init_models():
    res = [
        # RandomForestRegressorModel(100, None, 1),
        # DecisionTreeRegressorModel('squared_error', None, 1),
        # GradientBoostingRegressorModel('friedman_mse', 100, 0.1, 3, 1),
        XGBoostingRegressorModel('dart', 200, 0.1,  5),
    ]
    # for n_estimators in [100, 200, 400]:
    #     for max_depth in [11, 12, 13]:
    #         for min_samples_leaf in [1, 2]:
    #             res.append(RandomForestRegressorModel(n_estimators, max_depth, min_samples_leaf))
    return res

train_test_sets = {}

for add_candidate in add_candidates:
    if add_candidate in included_column_names:
        continue
    included_column_names.append(add_candidate)

    xs, ys, xs_submission, ids_submission = prepare_datasets(train, submission)
    xs = xs.to_numpy()
    xs_submission = xs_submission.to_numpy()

    x_train, x_test, y_train, y_test = \
        train_test_split(xs, ys.to_numpy(), test_size=0.05, random_state=42)

    for model in init_models():
        set_name = f'skip: {add_candidate}, model: {model.name()}'
        # print(set_name)
        train_test_sets[set_name] = [model, x_train, x_test, y_train, y_test, xs_submission, ids_submission]

    included_column_names = included_column_names[:-1]

best_name, error = test_models(train_test_sets)

model, _, _, _, _, xs_submission, ids_submission = train_test_sets[best_name]
write_best(model, error, xs_submission, ids_submission.to_numpy())


Площадь_общая    float64
lat              float64
lon              float64
dtype: object

       Площадь_общая         lat         lon
count    1017.000000  945.000000  945.000000
mean       61.479115   57.995932   43.894577
std        49.122761    1.728288   14.161377
min        10.000000   55.668955   30.108824
25%        36.000000   56.790941   30.338660
50%        45.000000   56.853332   37.579155
75%        65.000000   59.926728   60.586109
max       433.000000   60.058184   60.817212

count    1.017000e+03
mean     1.358026e+05
std      3.034808e+05
min      1.600000e+04
10%      2.800000e+04
20%      3.410000e+04
50%      5.000000e+04
70%      7.500000e+04
90%      3.080000e+05
95%      5.500000e+05
99%      1.284000e+06
max      3.700000e+06
Name: Price, dtype: float64

skip: nothing, model: xgboost_dart_200_0.1_5: error = 22175.380859

skip: nothing, model: xgboost_dart_200_0.1_5: error = 22175.380859

best: skip: nothing, model: xgboost_dart_200_0.1_5, error = 22175.3809
